In [0]:
# Databricks Notebook: DataLoad_Notebook
# Cell 1: Define widgets and validate inputs

dbutils.widgets.text("source_table", "", "Source Table Name")
dbutils.widgets.text("target_catalog", "bankingpoc", "Target Catalog")
dbutils.widgets.text("target_schema", "bronze", "Target Schema")
dbutils.widgets.text("target_table", "", "Target Table Name")
dbutils.widgets.text("adls_path", "", "ADLS Parquet Staging Path")
dbutils.widgets.text("load_type", "Full", "Load Type (Full/Incremental)")

source_table = dbutils.widgets.get("source_table").strip()
target_catalog = dbutils.widgets.get("target_catalog").strip()
target_schema = dbutils.widgets.get("target_schema").strip()
target_table = dbutils.widgets.get("target_table").strip()
adls_path = dbutils.widgets.get("adls_path").strip()
load_type = dbutils.widgets.get("load_type").strip()

# Parameter validation check
missing_params = []
if not target_table: missing_params.append("target_table")
if not adls_path: missing_params.append("adls_path")
if not target_catalog: missing_params.append("target_catalog")
if not target_schema: missing_params.append("target_schema")

if missing_params:
    error_msg = f"Missing required parameter(s): {', '.join(missing_params)}. Please provide values before running."
    print(f"ERROR: {error_msg}")
    dbutils.notebook.exit(f"FAILED: {error_msg}")

print(f"Target Destination: {target_catalog}.{target_schema}.{target_table}")
print(f"Reading from: {adls_path}")
print(f"Load Type: {load_type}")

Target Destination: bankingpoc.bronze.account
Reading from: abfss://lakehouse@adlsbankingpoc2026.dfs.core.windows.net/landing/branch/
Load Type: Full


In [0]:
# Cell 2: Load raw staging data and append Bronze tracking columns
from pyspark.sql.functions import current_timestamp, lit
from pyspark.sql.utils import AnalysisException

try:
    # Read the raw parquet extract written to ADLS landing directory by ADF
    df_raw = spark.read.parquet(adls_path)
    
    # Tag rows with mandatory tracking metadata without altering source values
    df_bronze = df_raw \
        .withColumn("_source_system", lit("ONPREM_SQLSERVER_BANKING")) \
        .withColumn("_load_type", lit(load_type.upper())) \
        .withColumn("_ingestion_timestamp", current_timestamp())

except AnalysisException as ae:
    error_msg = f"Failed to read parquet data from path {adls_path}: {str(ae)}"
    print(error_msg)
    dbutils.notebook.exit(f"FAILED: {error_msg}")

In [0]:
# Cell 3: Write processed DataFrame into Delta Lake table under bankingpoc.bronze

full_target_table = f"{target_catalog}.{target_schema}.{target_table}"
write_error = None

try:
    if load_type.lower() == "full":
        (df_bronze.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(full_target_table))
        print(f"Successfully overwrote table {full_target_table} with {df_bronze.count()} records.")
    
    else:
        (df_bronze.write
            .format("delta")
            .mode("append")
            .saveAsTable(full_target_table))
        print(f"Successfully appended {df_bronze.count()} records to {full_target_table}.")

except Exception as ex:
    write_error = f"Write operation failed for {full_target_table}: {str(ex)}"
    print(write_error)

# Place exit calls outside the try-except scope
if write_error:
    dbutils.notebook.exit(f"FAILED: {write_error}")
else:
    dbutils.notebook.exit("SUCCESS")